# Visualização interativa com ipyaladin

Este teste coloca lado a lado as camadas astronômicas da região do WiseView: um mosaico de uma visita feito a partir dos FITS SPHEREx reais, as posições do catálogo Gaia DR3 e o fundo DSS2 disponível pelo Aladin. Arraste/zoom no mapa e clique nos pontos Gaia para explorar a região.

**Importante:** os pontos Gaia são posições de catálogo, não fotos do Gaia. A camada SPHEREx é uma observação real de uma época; este mapa não é uma animação temporal. O Aladin ajuda na exploração e no contexto do campo, enquanto o GIF temporal continua sendo o produto separado. Gere os mosaicos antes de abrir este notebook com `uv run spherex-tracking timelapse`.

In [ ]:
from pathlib import Path
import json
from astropy.io import fits
from astropy.table import Table
from ipyaladin import Aladin

ROOT = Path.cwd()
if not (ROOT / "data/gaia_timelapse/gaia_dr3_sources.ecsv").exists():
    ROOT = ROOT.parent
assert (ROOT / "data/gaia_timelapse/gaia_dr3_sources.ecsv").exists(), "Execute o notebook a partir da raiz do projeto."

RA, DEC = 133.786245, -7.244372
FOV_ARCSEC = 176.0
catalog = Table.read(ROOT / "data/gaia_timelapse/gaia_dr3_sources.ecsv", format="ascii.ecsv")
report = json.loads((ROOT / "outputs/timelapse/timelapse_report.json").read_text())
mosaic_path = ROOT / report["frame_records"][0]["mosaic"]
assert mosaic_path.exists(), "Mosaicos ausentes. Rode `uv run spherex-tracking timelapse`."
print(f"Gaia DR3: {len(catalog)} fontes; mosaico SPHEREx: {mosaic_path.name}")

## Mapa

O DSS2 é uma imagem de referência para orientação visual. A imagem sobreposta é o mosaico de uma visita SPHEREx, gerado reprojetando as exposições para uma grade celeste comum com `reproject_exact` e combinando os pixels válidos com pesos da variância. Os marcadores amarelos mostram as posições Gaia DR3. A diferença de resolução e de comprimento de onda pode fazer as fontes não parecerem iguais entre as camadas.

In [ ]:
# Carrega o mosaico FITS produzido pelo pipeline. O FITS já tem WCS celeste comum.
with fits.open(mosaic_path) as mosaic_hdul:
    image_hdul_for_aladin = fits.HDUList([mosaic_hdul[0].copy()])

sky = Aladin(
    target=f"{RA} {DEC}",
    fov=FOV_ARCSEC / 3600,
    survey="P/DSS2/color",
    reticle=True,
)
sky.add_fits(image_hdul_for_aladin, name="Mosaico SPHEREx • visita 1", opacity=0.8)
sky.add_table(
    catalog,
    ra_field="ra",
    dec_field="dec",
    name="Gaia DR3 (catálogo)",
    color="yellow",
    sourceSize=10,
    shape="circle",
)
sky

## O que este teste responde

- **Ajuda** a explorar o campo com zoom/pan, ver posições Gaia por cima do céu e conferir onde o recorte SPHEREx cai.
- **Não fornece fotos Gaia**: DR3 não tem imagens por época para esta região.
- **Não substitui o GIF**: para comparar épocas, use `outputs/timelapse/spherex_wiseview.gif`; este widget é um mapa interativo de contexto.

Requer Jupyter/VS Code Notebook com widgets habilitados e internet para carregar o mapa e as imagens de fundo.

# GIF temporal: a mesma banda nas diferentes imagens

O eixo da animação é a **data da observação**. A banda 99 (4,851–4,893 µm; centro 4,873 µm) fica fixa. Cada frame vem de uma exposição diferente do campo WiseView (RA 133,786245°, Dec −7,244372°). A tabela de canais do cubo define a banda; os pixels vêm das exposições deste campo.

Nos 40 FITS locais, três imagens têm pixels válidos nessa banda. A cobertura é parcial, sobretudo na terceira imagem (cerca de 10% do campo), e aparece no título. Azul indica ausência de cobertura e não uma fonte desaparecendo. A escala de brilho é fixa.

Execute `uv run --extra viz spherex-tracking temporal-band` para refazer. A animação espectral anterior foi uma interpretação incorreta do pedido e permanece apenas como produto auxiliar.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent
display(Image(filename=str(ROOT / "outputs/temporal_band/spherex_fixed_band_observations.gif")))

In [ ]:
from spherex_tracking.temporal_band import temporal_viewer

viewer = temporal_viewer(ROOT / "outputs/temporal_band/report.json")
viewer